In [0]:
import json

CATALOG = "unified_lakehouse"
VOLUME_NAME = "report"
tables = [
    "bronze.taxi", "bronze.users", "bronze.drivers",
    "silver.taxi", "silver.drivers", "silver.users",
    "gold.taxi_metrics", "gold.driver_metrics",
    "gold.user_metrics", "gold.enriched_trips",
]

report = {}
for t in tables:
    try:
        history_df = spark.sql(f"DESCRIBE HISTORY {CATALOG}.{t}")
        history_records = history_df.collect()
        
        versions_history = []
        for row in history_records:
            version_num = row["version"]
            version_count = spark.read.option("versionAsOf", version_num).table(f"{CATALOG}.{t}").count()
            
            versions_history.append({
                "version": version_num,
                "operation": row["operation"],
                "row_count_at_version": version_count
            })
            
        report[t] = {
            "current_row_count": spark.table(f"{CATALOG}.{t}").count(),
            "execution_state": "SUCCESS",
            "history": versions_history
        }
    except Exception as e:
        report[t] = {
            "current_row_count": 0,
            "execution_state": f"FAILED: {str(e)}",
            "history": []
        }

output_path = f"/Volumes/{CATALOG}/raw_files/{VOLUME_NAME}/workflow_report.json"
dbutils.fs.put(output_path, json.dumps(report, indent=2), overwrite=True)